# 실습 3: AgentCore Gateway를 사용하여 에이전트에 도구를 안전하게 연결하기

## 개요

이 실습에서는 Amazon Bedrock Gateway를 사용하여 조직에서 제공하는 도구를 고객 지원 에이전트와 통합하는 방법을 알아봅니다.

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro)는 애플리케이션이 Large Language Models (LLMs)에 도구와 컨텍스트를 제공하는 방식을 표준화한 개방형 프로토콜입니다.

[Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)를 사용하면 개발자는 몇 줄의 코드만으로 API, Lambda 함수, 기존 서비스를 MCP 호환 도구로 변환하고 Gateway 엔드포인트를 통해 에이전트에 제공할 수 있습니다.


**워크숍 과정:**

- **실습 1 (완료):** 에이전트 프로토타입 만들기 - 실제로 작동하는 고객 지원 에이전트 구축
- **실습 2 (완료):** 메모리로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3 (현재):** Gateway 및 Identity로 확장 - 여러 에이전트에서 도구를 안전하게 공유
- **실습 4:** 프로덕션에 배포 - 관찰 기능과 함께 AgentCore Runtime 사용
- **실습 5:** 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기


### AgentCore Gateway와 도구 공유가 중요한 이유

현재 상태(실습 1~2): 각 에이전트가 자체 도구 사본을 보유하고 있습니다. 이 방식은 확장하기 어려우며 다음 문제를 일으킵니다.

- 서로 다른 에이전트 간 코드 중복
- 일관되지 않은 도구 동작 및 유지 관리 부담
- 중앙 집중식 보안 또는 액세스 제어 부재
- 여러 사용 사례로 확장하기 어려움

이 실습을 마치면 다음 에이전트에서 사용할 수 있는 중앙 집중식 재사용 도구를 갖추게 됩니다.

- 고객 지원 에이전트(현재 사용 사례)
- 영업 에이전트(동일한 제품 정보 및 고객 데이터 필요)
- 재고 에이전트(동일한 제품 정보 및 보증 확인 필요)
- 반품 처리 에이전트(반품 정책 및 고객 프로필 필요)

그 밖의 사용 사례에도 활용할 수 있습니다.

### AgentCore Identity를 사용하여 안전한 인증 추가

또한 AgentCore Gateway에서는 inbound 및 outbound 연결을 모두 안전하게 인증해야 합니다. [AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)는 Okta, Entra, Amazon Cognito와 같은 표준 identity provider를 지원하면서 AWS 서비스와 Slack, Zoom 같은 서드 파티 애플리케이션 전반에서 원활한 에이전트 identity 및 액세스 관리를 제공합니다. 이 실습에서는 AgentCore Gateway와 AgentCore Identity를 통합하여 inbound 및 outbound 인증을 통해 안전하게 연결하는 방법을 살펴봅니다.

Inbound 인증에서 AgentCore Gateway는 호출 시 전달된 OAuth 토큰을 분석하여 Gateway의 도구에 대한 액세스를 허용하거나 거부합니다. 도구가 외부 리소스에 액세스해야 하는 경우 AgentCore Gateway는 API Key, IAM 또는 OAuth Token을 통한 outbound 인증으로 외부 리소스에 대한 액세스를 허용하거나 거부할 수 있습니다.

Inbound 권한 부여 흐름에서는 에이전트 또는 MCP 클라이언트가 사용자의 IdP에서 생성된 OAuth 액세스 토큰을 추가하여 AgentCore Gateway의 MCP 도구를 호출합니다. 그러면 AgentCore Gateway가 OAuth 액세스 토큰을 검증하고 inbound 권한 부여를 수행합니다.

AgentCore Gateway에서 실행되는 도구가 외부 리소스에 액세스해야 하는 경우 OAuth는 Gateway target의 resource credential provider를 사용하여 downstream 리소스의 자격 증명을 검색합니다. AgentCore Gateway는 downstream API에 액세스할 수 있도록 호출자에게 권한 부여 자격 증명을 전달합니다.


## 실습 3 아키텍처

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway.png" width="75%"/>
</div>

*이제 웹 검색 도구가 안전한 identity 기반 액세스 제어와 함께 AgentCore Gateway에 중앙 집중화됩니다. 여러 에이전트와 사용 사례에서 동일한 도구를 안전하게 공유할 수 있습니다. 다른 애플리케이션용으로 구축한 `check_warranty()` 도구도 재사용하고, 다른 애플리케이션에서 사용할 수 있도록 `web_search()` 도구를 추가합니다. `get_product_info()`, `get_return_policy()`, `get_technical_support`는 고객 지원 사용 사례에 특화되어 있으므로 로컬 도구로 유지합니다.*

### 주요 기능
- **AWS Lambda 함수의 원활한 통합:** 이 예제에서는 Amazon Bedrock AgentCore Gateway를 사용하여 에이전트를 기존 AWS Lambda 함수와 통합하고, 제품 보증을 확인하고 고객 프로필을 가져오는 방법을 보여 줍니다.
- **Inbound Auth로 Gateway 엔드포인트 보호:** 유효한 JWT 토큰을 제공하는 에이전트만 엔드포인트에 연결하여 도구를 사용할 수 있습니다.
- **MCP 엔드포인트를 사용하도록 에이전트 구성:** 에이전트가 유효한 JWT 토큰을 가져와 AgentCore Gateway에서 제공하는 MCP 엔드포인트에 연결합니다.

## 사전 요구 사항

* Python 3.12 이상
* AWS 자격 증명 구성
* 실습 2 '고객 지원 에이전트에 메모리 추가' 완료
* AWS 워크숍 계정에는 다음 리소스가 미리 생성되어 있습니다.
    - AWS Lambda 함수
    - AWS Lambda Execution IAM Role
    - AgentCore Gateway IAM Role
    - AWS Lambda 함수에서 사용하는 DynamoDB 테이블
    - Cognito User Pool 및 User Pool Client

## 단계 1: 필수 라이브러리 가져오기

In [ ]:
# 라이브러리 가져오기
import os
import sys
import boto3
import json
import uuid
import time

from lab_helpers.utils import suppress_warnings

suppress_warnings()

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

import asyncio
import nest_asyncio

nest_asyncio.apply()

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from bedrock_agentcore.memory import MemoryClient
from lab_helpers.lab2_memory import ACTOR_ID, create_or_get_memory_resource

from lab_helpers.utils import (
    get_or_create_cognito_pool,
    put_ssm_parameter,
    get_ssm_parameter,
    load_api_spec,
)


sts_client = boto3.client("sts")
account_id = sts_client.get_caller_identity()["Account"]
# AWS 계정 세부 정보 가져오기
REGION = boto3.session.Session().region_name

gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=REGION,
)

print("\u2705 Libraries imported successfully!")

## 단계 2: 기존 고객 데이터에 액세스하는 도구를 에이전트에 제공
AgentCore Gateway는 다음 세 가지 주요 방식으로 에이전트 도구 통합을 간소화합니다.

범용 MCP 지원: AgentCore Gateway의 MCP 표준을 통해 도구를 제공하여 어떤 에이전트 프레임워크와도 즉시 호환

간단한 REST 통합: 기존 REST 서비스를 AgentCore Gateway target으로 추가하기만 하면 에이전트 도구로 변환

Lambda 유연성: 모든 API를 호출할 수 있는 MCP 엔드포인트로 Lambda 함수 제공 - 여기서는 보증 상태를 확인하는 함수로 시연

AgentCore Gateway는 Lambda context에 호출할 도구 이름을 채우고, 도구에 전달할 파라미터는 Lambda event에 제공합니다.

```
extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
resource = extended_tool_name.split("___")[1]
```

[Lambda 함수](./prerequisite/lambda/python/lambda_function.py)

```
def lambda_handler(event, context):
    if get_tool_name(event) == "check_warranty_status":
        serial_number = get_named_parameter(event=event, name="serial_number")
        customer_email = get_named_parameter(event=event, name="customer_email")

        warranty_status = check_warranty_status(serial_number, customer_email)
        return {"statusCode": 200, "body": warranty_status}
```

## 단계 3: 웹 검색 도구를 MCP로 변환
이제 AgentCore Gateway를 사용하여 MCP 서버를 개발하고 있으므로 여러 에이전트에서 사용할 도구를 MCP와 호환되게 만들 수 있습니다. 이러한 도구 중 하나로 실습 1에서 구축한 웹 검색 도구를 활용할 수 있습니다. 이에 따라 실습 1의 웹 검색 도구도 AgentCore Gateway 내의 Lambda 도구로 변환했습니다.

[웹 검색 Lambda](./prerequisite/lambda/python/web_search.py)
```
from ddgs import DDGS


def web_search(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc.
        max_results (int): The maximum number of results to return.
        
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"


print("\u2705 Web search tool ready")
```

## 단계 4: 함수 정의 metadata 생성
마지막으로 Lambda 함수에서 구현한 도구를 설명하는 tool schema를 작성해야 합니다.

이 파일은 [prerequisite/lambda/api_spec.json](./prerequisite/lambda/api_spec.json)에 이미 정의되어 있습니다.

```
[
    {
        "name": "check_warranty_status",
        "description": "Check the warranty status of a product using its serial number and optionally verify via email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {
                    "type": "string"
                },
                "customer_email": {
                    "type": "string"
                }
            },
            "required": [
                "serial_number"
            ]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information using DuckDuckGo",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {
                    "type": "string",
                    "description": "The search query keywords"
                },
                "region": {
                    "type": "string",
                    "description": "The search region (e.g., us-en, uk-en, ru-ru)"
                },
                "max_results": {
                    "type": "integer",
                    "description": "The maximum number of results to return"
                }
            },
            "required": [
                "keywords"
            ]
        }
    }
]
```

## 단계 5: AgentCore Gateway 생성

Lambda 함수를 MCP 호환 엔드포인트로 제공할 AgentCore Gateway를 생성합니다.

도구를 호출할 권한이 있는 호출자를 검증하려면 Inbound Auth를 구성해야 합니다.

Inbound Auth는 MCP 서버의 표준인 OAuth 권한 부여를 사용합니다. OAuth를 사용할 때 클라이언트 애플리케이션은 Gateway를 사용하기 전에 OAuth authorizer로 인증해야 합니다. 클라이언트는 런타임에 사용할 access token을 받습니다.

OAuth discovery server와 클라이언트 ID를 지정해야 합니다. 워크숍에서 제공하는 CloudFormation은 Cognito UserPool과 UserPoolClient를 이미 프로비저닝했으며, discovery URL과 Client ID를 전용 SSM 파라미터에 저장했습니다.

In [ ]:
gateway_name = "customersupport-gw"

cognito_config = get_or_create_cognito_pool(refresh_token=True)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

try:
    # 새 Gateway 생성
    print(f"Creating gateway in region {REGION} with name: {gateway_name}")

    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=auth_config,
        description="Customer Support AgentCore Gateway",
    )

    gateway_id = create_response["gatewayId"]

    gateway = {
        "id": gateway_id,
        "name": gateway_name,
        "gateway_url": create_response["gatewayUrl"],
        "gateway_arn": create_response["gatewayArn"],
    }
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_name", gateway_name)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_arn", create_response["gatewayArn"])
    put_ssm_parameter("/app/customersupport/agentcore/gateway_url", create_response["gatewayUrl"])
    print(f"\u2705 Gateway created successfully with ID: {gateway_id}")

except Exception:
    # Gateway가 있으면 SSM에서 기존 Gateway ID 가져오기
    existing_gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    print(f"Found existing gateway with ID: {existing_gateway_id}")

    # 기존 Gateway 세부 정보 가져오기
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=existing_gateway_id)
    gateway = {
        "id": existing_gateway_id,
        "name": gateway_response["name"],
        "gateway_url": gateway_response["gatewayUrl"],
        "gateway_arn": gateway_response["gatewayArn"],
    }
    gateway_id = gateway["id"]

## 단계 6: Lambda 함수 Target 추가
이제 [prerequisite/lambda/api_spec.json](./prerequisite/lambda/api_spec.json)에 앞서 정의한 함수 정의를 사용하여 Agent Gateway 내에 Lambda target을 생성합니다. 이를 통해 Gateway에서 호스팅할 도구를 정의합니다.

Gateway에는 여러 target을 연결할 수 있으며, Gateway에 연결된 target 또는 도구는 언제든 변경할 수 있습니다. 각 target은 자체 credential provider를 사용할 수 있지만, Gateway는 다양한 API 전반에서 에이전트에 필요한 모든 도구에 액세스할 수 있는 단일 MCP URL을 제공합니다.

In [ ]:
try:
    api_spec_file = "./prerequisite/lambda/api_spec.json"

    # API spec 파일이 있는지 확인
    if not os.path.exists(api_spec_file):
        print(f"\u274c API specification file not found: {api_spec_file}")
        sys.exit(1)

    api_spec = load_api_spec(api_spec_file)

    # Gateway의 Inbound OAuth에 Cognito 사용
    lambda_target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
                "toolSchema": {"inlinePayload": api_spec},
            }
        }
    }

    # Gateway target 생성
    credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaUsingSDK",
        description="Lambda Target using SDK",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=credential_config,
    )

    print(f"\u2705 Gateway target created: {create_target_response['targetId']}")

except Exception as e:
    print(f"\u274c Error creating gateway target: {str(e)}")

## 단계 7: 새로운 MCP 기반 도구를 지원 에이전트에 추가
여기서는 Cognito의 인증 토큰을 MCP 클라이언트와 통합하여 AgentCore Gateway에 연결한 다음, MCP 도구를 Google ADK 호환 함수로 래핑합니다.
### 단계 7.1: 안전한 MCP 클라이언트 설정 및 사용 가능한 도구 나열

In [ ]:
print(f"Gateway Endpoint - MCP URL: {gateway['gateway_url']}")


# MCP 클라이언트를 직접 사용하여 Gateway에서 사용 가능한 도구 나열
async def list_gateway_tools():
    async with streamablehttp_client(
        gateway["gateway_url"],
        headers={"Authorization": f"Bearer {cognito_config['bearer_token']}"},
    ) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.list_tools()
            print(f"   Found {len(result.tools)} tool(s):\n")
            for tool in result.tools:
                print(f"   \u2705 {tool.name}")
                print(f"      {tool.description}\n")
            return result.tools


gateway_tools_list = await list_gateway_tools()

### 단계 7.2: 로컬 도구 + MCP Gateway 도구 + 메모리를 사용하는 에이전트 생성

다음 요소를 결합하여 Google ADK 에이전트를 생성합니다.
- **로컬 도구**: `get_product_info`, `get_return_policy`, `get_technical_support`(고객 지원에 특화)
- **AgentCore Gateway를 통한 MCP 도구**: `check_warranty_status`, `web_search`(여러 에이전트에서 공유)
- **AgentCore Memory**: 명시적인 메모리 검색 및 저장(실습 2와 동일한 패턴)

Google ADK는 MCP 도구를 기본으로 사용하지 않으므로 Gateway MCP 엔드포인트를 호출하는 간단한 래퍼 함수를 생성합니다. 이러한 래퍼는 일반 ADK 도구로 등록됩니다.

In [ ]:
# ============================================================
# 로컬 도구(실습 1/2와 동일하며 고객 지원에 특화)
# ============================================================


def get_return_policy(product_category: str) -> str:
    """Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available",
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty",
        },
    }
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection",
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies",
    }
    policy = return_policies.get(product_category.lower(), default_policy)
    return (
        f"Return Policy - {product_category.title()}:\n\n"
        f"\u2022 Return window: {policy['window']} from delivery\n"
        f"\u2022 Condition: {policy['condition']}\n"
        f"\u2022 Process: {policy['process']}\n"
        f"\u2022 Refund timeline: {policy['refund_time']}\n"
        f"\u2022 Shipping: {policy['shipping']}\n"
        f"\u2022 Warranty: {policy['warranty']}"
    )


def get_product_info(product_type: str) -> str:
    """Get detailed technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')
    Returns:
        Formatted product information including warranty, features, and policies
    """
    products = {
        "laptops": {
            "warranty": "1-year standard, 3-year extended available",
            "specs": "Intel/AMD processors, 8-64GB RAM, SSD storage",
            "features": "Backlit keyboards, fingerprint readers, Thunderbolt ports",
            "compatibility": "Windows, Linux, macOS (Apple only)",
            "support": "24/7 technical support, on-site repair options",
        },
        "smartphones": {
            "warranty": "1-year manufacturer, 2-year extended",
            "specs": "Latest processors, 6-12GB RAM, 128GB-1TB storage",
            "features": "5G capable, water resistant, wireless charging",
            "compatibility": "iOS or Android ecosystem",
            "support": "In-store and mail-in repair services",
        },
        "headphones": {
            "warranty": "1-year standard warranty",
            "specs": "Bluetooth 5.0+, ANC, 20-40hr battery",
            "features": "Active noise cancellation, transparency mode, multipoint",
            "compatibility": "Universal Bluetooth, some with proprietary apps",
            "support": "Replacement program for defective units",
        },
        "monitors": {
            "warranty": "3-year standard, zero dead pixel guarantee",
            "specs": "4K/1440p resolution, 60-240Hz refresh rate",
            "features": "HDR support, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
            "support": "Color calibration and technical support",
        },
    }
    product = products.get(product_type.lower())
    if not product:
        return f"Technical specifications for {product_type} not available. Please contact our technical support team."
    return (
        f"Technical Information - {product_type.title()}:\n\n"
        f"\u2022 Warranty: {product['warranty']}\n"
        f"\u2022 Specifications: {product['specs']}\n"
        f"\u2022 Key Features: {product['features']}\n"
        f"\u2022 Compatibility: {product['compatibility']}\n"
        f"\u2022 Support: {product['support']}"
    )


def get_technical_support(issue_description: str) -> str:
    """Search the technical support knowledge base for troubleshooting help, setup guides, and maintenance tips.

    Args:
        issue_description: Description of the technical issue or question the customer needs help with.
    Returns:
        Relevant technical support documentation and troubleshooting steps.
    """
    try:
        ssm = boto3.client("ssm")
        acct = boto3.client("sts").get_caller_identity()["Account"]
        region = boto3.Session().region_name
        kb_id = ssm.get_parameter(Name=f"/{acct}-{region}/kb/knowledge-base-id")["Parameter"]["Value"]
        bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=region)
        response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={"text": issue_description},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
        )
        results = response.get("retrievalResults", [])
        if not results:
            return "No relevant technical support documentation found for this issue."
        formatted_results = []
        for i, result in enumerate(results, 1):
            text = result.get("content", {}).get("text", "")
            score = result.get("score", 0)
            if score >= 0.4:
                formatted_results.append(f"--- Result {i} (relevance: {score:.2f}) ---\n{text}")
        if not formatted_results:
            return "No sufficiently relevant technical support documentation found."
        return "\n\n".join(formatted_results)
    except Exception as e:
        return f"Unable to access technical support documentation. Error: {str(e)}"


print("\u2705 Local tools defined")

In [ ]:
# ============================================================
# MCP Gateway 도구 래퍼
# Gateway MCP 엔드포인트를 호출하는 간단한 래퍼 함수입니다.
# Gateway 도구를 ADK에 일반 Python 함수로 제공합니다.
# ============================================================

import nest_asyncio

nest_asyncio.apply()


async def _call_mcp_tool(tool_name: str, arguments: dict) -> str:
    """AgentCore Gateway의 MCP 도구를 호출하는 헬퍼입니다."""
    async with streamablehttp_client(
        gateway["gateway_url"],
        headers={"Authorization": f"Bearer {cognito_config['bearer_token']}"},
    ) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            # MCP 결과에서 텍스트 콘텐츠 추출
            if result.content:
                return "\n".join(part.text for part in result.content if hasattr(part, "text"))
            return "No result returned."


def check_warranty_status(serial_number: str, customer_email: str) -> str:
    """Check the warranty status of a product using its serial number and optionally verify via email.

    Args:
        serial_number: The product serial number to look up.
        customer_email: Customer email for verification. Pass empty string if not available.

    Returns:
        Warranty status information for the product.
    """
    args = {"serial_number": serial_number}
    if customer_email:
        args["customer_email"] = customer_email
    return asyncio.get_event_loop().run_until_complete(_call_mcp_tool("LambdaUsingSDK___check_warranty_status", args))


def web_search(keywords: str, region: str, max_results: int) -> str:
    """Search the web for updated information using DuckDuckGo.

    Args:
        keywords: The search query keywords.
        region: The search region (e.g., us-en, uk-en, ru-ru).
        max_results: The maximum number of results to return.

    Returns:
        Search results from the web.
    """
    args = {"keywords": keywords, "region": region, "max_results": max_results}
    return asyncio.get_event_loop().run_until_complete(_call_mcp_tool("LambdaUsingSDK___web_search", args))


print("\u2705 MCP gateway tool wrappers defined")

In [ ]:
# ============================================================
# Google ADK + Memory 통합을 사용한 에이전트 생성
# ============================================================

SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications.
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. get_technical_support() - To search the technical support knowledge base
4. check_warranty_status() - To check warranty status via serial number (via AgentCore Gateway)
5. web_search() - To access current technical documentation, or for updated information (via AgentCore Gateway)
Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications."""


# 모든 도구를 사용하는 Google ADK 에이전트 생성
all_tools = [
    get_product_info,
    get_return_policy,
    get_technical_support,
    check_warranty_status,
    web_search,
]

agent = LlmAgent(
    name="customer_support_agent",
    model=LiteLlm(model="bedrock/us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    instruction=SYSTEM_PROMPT,
    tools=all_tools,
)

# Memory 설정
memory_id = create_or_get_memory_resource()
memory_client = MemoryClient(region_name=REGION)

APP_NAME = "customer_support_app"
USER_ID = "user_001"


async def create_agent(prompt):
    """Memory 컨텍스트를 주입하여 에이전트에 질의를 보냅니다."""
    session_id = str(uuid.uuid4())

    # --- 1. Memory에서 고객 컨텍스트 검색 ---
    all_context = []
    namespaces = {
        "preferences": f"support/customer/{ACTOR_ID}/preferences/",
        "semantic": f"support/customer/{ACTOR_ID}/semantic/",
    }
    for context_type, namespace in namespaces.items():
        try:
            memories = memory_client.retrieve_memories(
                memory_id=memory_id,
                namespace=namespace,
                query=prompt,
                top_k=3,
            )
            for mem in memories:
                if isinstance(mem, dict):
                    text = mem.get("content", {}).get("text", "").strip()
                    if text:
                        all_context.append(f"[{context_type.upper()}] {text}")
        except Exception as e:
            print(f"Warning: Could not retrieve {context_type} memories: {e}")

    # --- 2. 컨텍스트로 보강된 질의 구성 ---
    if all_context:
        context_text = "\n".join(all_context)
        enriched_query = f"Customer Context:\n{context_text}\n\n{prompt}"
        print(f"\U0001f4cb Retrieved {len(all_context)} memory items for context")
    else:
        enriched_query = prompt
        print("No prior memory context found")

    # --- 3. ADK 에이전트 실행 ---
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=enriched_query)])

    final_response = ""
    async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=content):
        if event.is_final_response():
            final_response = event.content.parts[0].text

    # --- 4. 상호 작용을 Memory에 저장 ---
    if final_response:
        try:
            memory_client.create_event(
                memory_id=memory_id,
                actor_id=ACTOR_ID,
                session_id=session_id,
                messages=[
                    (prompt, "USER"),
                    (final_response, "ASSISTANT"),
                ],
            )
            print("\U0001f4be Interaction saved to memory")
        except Exception as e:
            print(f"Warning: Could not save to memory: {e}")

    return final_response


print("\u2705 Customer support agent created successfully!")

## 단계 8: 기존 API에 대한 MCP 도구 액세스로 에이전트 테스트

모든 기능이 올바르게 작동하는지 샘플 질의로 에이전트를 테스트합니다. 출력에 5개 도구가 표시되어야 합니다.

In [ ]:
test_prompts = [
    # 보증 확인
    "List all of your tools",
    "I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?",
    "I have a Gaming Console Pro device , I want to check my warranty status, warranty serial number is MNO33333333.",
    "What are the warranty support guidelines?",
    "How can I fix Lenovo Thinkpad with a blue screen",
    "Tell me detailed information about the technical documentation on installing a new CPU",
]


# 에이전트 테스트 함수
async def test_agent_responses(prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = await create_agent(prompt)
            print(response)
        except Exception as e:
            # 중첩된 ExceptionGroup에서 근본 원인 추출
            root = e
            while hasattr(root, "exceptions") and root.exceptions:
                root = root.exceptions[0]
            print(f"Error: {root}")
        print("-" * 50)


# 테스트 실행
await test_agent_responses(test_prompts)

print("\n\u2705 Basic testing completed!")

## [선택 사항] AgentCore Policy

### 단계 9: Policy Engine 생성

세분화된 액세스 제어를 위한 Cedar 권한 부여 정책을 포함하는 Policy Engine을 생성합니다.

In [ ]:
# toolkit에서 가져오고, 사용할 수 없으면 사용자 지정 구현 사용
try:
    from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

    print("\u2705 Using toolkit PolicyClient")
except ImportError:
    from utils.policy_utils import PolicyClient

    print("\u2705 Using custom PolicyClient (toolkit policy module not available)")

# Policy 클라이언트 초기화
policy_client = PolicyClient(region_name=REGION)

print("\n\U0001f527 Creating Policy Engine...")

# Policy Engine을 생성하거나 기존 항목 가져오기
# Policy Engine은 모든 권한 부여 정책을 담는 컨테이너입니다.
engine = policy_client.create_or_get_policy_engine(
    name="customersupport_pe",
    description="Policy engine for customer support gateway",
)

engine_id = engine["policyEngineId"]
engine_arn = engine["policyEngineArn"]
put_ssm_parameter("/app/customersupport/agentcore/policy_engine_id", engine_id)

print("\n\u2705 Policy Engine ready")
print(f"   Engine ID: {engine_id}")
print(f"   Engine ARN: {engine_arn}")

### [선택 사항]: 자연어로 Cedar Policy 생성

AgentCore Policy는 자연어를 사용하여 Cedar Policy를 생성하고 범위 기반 액세스를 적용하는 기능을 제공합니다.

In [ ]:
# approve 도구용 Cedar policy 생성(write scope)
print("\n\U0001f4dd Generating Cedar Policy from Natural language...")

nl_input = "Allow tag username == 'testuser' to perform check warranty status on the customer support gateway."

warranty_tool_policy = policy_client.generate_policy(
    policy_engine_id=engine["policyEngineId"],
    name=f"nl_policy_{int(time.time())}",
    resource={"arn": gateway["gateway_arn"]},
    content={"rawText": nl_input},
    fetch_assets=True,
)

print("\u2705 Policy generated from natural language")

In [ ]:
print("\U0001f4cb Generated Cedar Policies:\n")
print("=" * 80)

# 보증 상태 허용 정책
print("\n1\ufe0f\u20e3  Warranty Status")
print("-" * 80)
warranty_tool_policy_cedar = warranty_tool_policy["generatedPolicies"][0]["definition"]["cedar"]["statement"]
print(warranty_tool_policy_cedar)

print("\n" + "=" * 80)

### 단계 10: Cedar Policy 생성

범위 기반 액세스를 적용할 Cedar Policy를 생성합니다. 승인 작업에는 write 범위를, 생성/목록 작업에는 read 범위를 사용할 수 있습니다.

이 워크숍에서는 모든 사용자에게 일관된 결과를 제공하기 위해 미리 작성된 허용/거부 Cedar Policy를 사용합니다.

In [ ]:
# 사용 사례 실행의 일관성을 위한 정책 제공
allow_policy = {
    "cedar": {
        "statement": f"""permit(
            principal,
            action in [AgentCore::Action::"LambdaUsingSDK___check_warranty_status", AgentCore::Action::"LambdaUsingSDK___web_search"],
            resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/{gateway["id"]}"
        ) when {{
            (principal.hasTag("username")) && 
            ((principal.getTag("username")) == "testuser")
        }};"""
    }
}

# "iPhone 8" 키워드에 대한 웹 검색 거부
deny_web_search_policy = {
    "cedar": {
        "statement": f"""forbid(
            principal,
            action == AgentCore::Action::"LambdaUsingSDK___web_search",
            resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/{gateway["id"]}"
        ) when {{
            context.input has keywords &&
            context.input.keywords like "*iPhone 8*"
        }};"""
    }
}

### 단계 11: Policy Engine에 정책 추가

생성된 Cedar Policy를 Policy Engine에 추가합니다.

In [ ]:
print("\U0001f527 Creating policies in Policy Engine...\n")

# 두 도구를 모두 허용하는 정책 생성
warranty_result = policy_client.create_or_get_policy(
    policy_engine_id=engine["policyEngineId"],
    name="allow_policy",
    description="Allow web_search and check_warranty_status calls",
    definition=allow_policy,
)
print("\u2705 Policy ready: allow_policy")
print("   Tools allowed: check_warranty_status and web_search\n")

# 웹 검색을 거부하는 목록 정책 생성
web_search_deny_result = policy_client.create_or_get_policy(
    policy_engine_id=engine["policyEngineId"],
    name="deny_web_search",
    description="Deny web_search tool call for iPhone 8",
    definition=deny_web_search_policy,
)
print("\u2705 Policy ready: deny_web_search")
print("   Tools denied conditionally: web_search\n")

print("\u2705 All policies ready!")

### 단계 12: Gateway IAM Role 권한 업데이트

Policy Engine을 연결하기 전에 Gateway IAM Role에 Policy Engine 액세스 권한을 부여해야 합니다.

In [ ]:
role_arn = get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role")
role_name = role_arn.split("/")[-1]

iam_client = boto3.client("iam")
print("\U0001f527 Updating Gateway IAM role with Policy Engine permissions...")

# Policy Engine 액세스 권한을 부여하는 정책 문서
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:*"],
            "Resource": [
                f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:policy-engine/*",
                f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*",
            ],
        }
    ],
}

try:
    # role에 inline policy 추가
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName="PolicyEngineAccess",
        PolicyDocument=json.dumps(policy_document),
    )

    print("\u2705 IAM role updated successfully")
    print(f"   Role: {role_name}")
    print("   Added permissions: GetPolicyEngine, GetPolicy, ListPolicies")
    print("\n\u23f3 Waiting 10 seconds for IAM changes to propagate...")
    time.sleep(10)

    print("\u2705 Ready to attach Policy Engine")

except Exception as e:
    print(f"\u274c Error updating IAM role: {e}")
    print("\nYou may need to manually add these permissions to the role.")

### 단계 13: Gateway에 Policy Engine 연결

Policy Engine을 ENFORCE 모드로 Gateway에 연결하여 정책 적용을 활성화합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient

# Gateway 클라이언트 초기화
gateway_client_toolkit = GatewayClient(region_name=REGION)

print("\U0001f527 Attaching Policy Engine to Gateway...")
print("   Mode: ENFORCE (policies will block unauthorized requests)\n")

# Policy Engine을 Gateway에 연결
update_response = gateway_client_toolkit.update_gateway_policy_engine(
    gateway_identifier=gateway["id"],
    policy_engine_arn=engine["policyEngineArn"],
    mode="ENFORCE",
)

print("\u2705 Policy Engine attached successfully!")
print(f"   Gateway ID: {gateway['id']}")
print(f"   Policy Engine: {engine['policyEngineId']}")
print("   Mode: ENFORCE")
print("\n\U0001f512 Authorization is now active!")

### 단계 14: 정책 적용 테스트

In [ ]:
test_prompts = [
    "List all of your tools",
    "Search the web for heating issues with Samsung zfold 7",
    "Search the internet for heating issues with iPhone 8",
]


# 에이전트 테스트 함수
async def test_agent_responses(prompts):
    for i, prompt in enumerate(prompts, 1):
        print(f"\nTest Case {i}: {prompt}")
        print("-" * 50)
        try:
            response = await create_agent(prompt)
            print(response)
        except Exception as e:
            # 중첩된 ExceptionGroup에서 근본 원인 추출
            root = e
            while hasattr(root, "exceptions") and root.exceptions:
                root = root.exceptions[0]
            print(f"Error: {root}")
        print("-" * 50)


# 테스트 실행
await test_agent_responses(test_prompts)

print("\n\u2705 Policy testing completed!")

### 축하합니다! 🎉


실습 3 'AgentCore Gateway를 사용하여 에이전트에 도구를 안전하게 연결하기'를 성공적으로 완료했습니다.

이번 실습에서 완료한 내용:

##### 도구 중앙 집중화 및 재사용성

- 웹 검색을 로컬 도구에서 중앙 집중식 AgentCore Gateway로 마이그레이션
- 기존 엔터프라이즈 Lambda 함수 통합(보증 확인, 고객 프로필)
- 여러 유형의 에이전트가 액세스할 수 있는 공유 도구 인프라 생성

##### 엔터프라이즈 수준의 보안

- Cognito 통합을 통한 JWT 기반 인증 구현
- Gateway 액세스를 위한 안전한 inbound 권한 부여 구성
- 도구 사용을 위한 identity 기반 액세스 제어 설정

##### 확장 가능한 아키텍처 기반

- 여러 사용 사례(고객 지원, 영업, 반품 처리)에 활용할 수 있는 재사용 도구 구축
- 서로 다른 에이전트 간 코드 중복 제거
- 도구 업데이트 및 유지 관리를 위한 중앙 집중식 관리 체계 구축

##### 현재 제한 사항(다음 실습에서 해결합니다)

- **로컬 개발 환경** - 여전히 노트북에서 실행되므로 프로덕션 준비가 되지 않았습니다.
- **제한된 관찰 기능** - 에이전트 동작 및 성능을 종합적으로 모니터링할 수 없습니다.
- **수동 확장** - 증가한 부하나 여러 동시 사용자를 자동으로 처리할 수 없습니다.

##### 다음 실습: [실습 4 - AgentCore Runtime을 사용하여 프로덕션에 배포하기 →](lab-04-agentcore-runtime.ipynb)

실습 4에서는 다음 기능을 통해 프로토타입을 프로덕션 준비 시스템으로 전환합니다.

- 확장 가능한 에이전트 배포를 위한 AgentCore Runtime
- 메트릭, 로깅, 추적을 활용한 종합적인 관찰 기능
- 실제 트래픽을 처리하는 auto scaling 기능

### 리소스
- [Amazon Bedrock Agent Core Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)
- [Google ADK Documentation](https://google.github.io/adk-docs/)
- [Google ADK with LiteLLM](https://google.github.io/adk-docs/agents/models/litellm/)
- [Official Customer Support Sample](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/02-use-cases/customer-support-assistant)